<a href="https://colab.research.google.com/github/pSenchua/Patient-health-assessment-app-with-server-integration./blob/main/Weather%2BEnergy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score,
    roc_curve
)

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("assam_weather_energy_compiled.csv")

df = df.drop_duplicates()
df = df.fillna(method='ffill').fillna(method='bfill')

# Remove date columns if exist
for col in df.columns:
    if "date" in col.lower():
        df = df.drop(columns=[col])

df = df.select_dtypes(include=[np.number])

print("Dataset shape:", df.shape)

Dataset shape: (3552, 15)


/tmp/ipython-input-1075367752.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill').fillna(method='bfill')


In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

threshold_energy = np.percentile(y, 70)
y_cls = (y >= threshold_energy).astype(int)

In [ ]:
selector = SelectKBest(score_func=f_regression, k=min(15, X.shape[1]))
X_selected = selector.fit_transform(X, y)

In [ ]:
X_train, X_test, y_train, y_test, y_train_cls, y_test_cls = train_test_split(
    X_selected, y, y_cls, test_size=0.25, random_state=42, stratify=y_cls
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
def train_autoencoder(model, name):

    early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    model.fit(
        X_train, X_train,
        epochs=150,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early],
        verbose=0
    )

    encoder = models.Model(model.input, model.layers[-2].output)

    X_train_enc = encoder.predict(X_train)
    X_test_enc = encoder.predict(X_test)

    print(f"{name} trained.")
    return X_train_enc, X_test_enc

In [ ]:
def basic_ae(input_dim):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    bottleneck = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(bottleneck)
    out = layers.Dense(input_dim)(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
def deep_ae(input_dim):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dense(64, activation='relu')(x)
    bottleneck = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(bottleneck)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(input_dim)(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
def denoise_ae(input_dim):
    inp = layers.Input(shape=(input_dim,))
    x = layers.GaussianNoise(0.2)(inp)
    x = layers.Dense(64, activation='relu')(x)
    bottleneck = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(bottleneck)
    out = layers.Dense(input_dim)(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
def sparse_ae(input_dim):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu',
                     activity_regularizer=regularizers.l1(1e-4))(inp)
    bottleneck = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(bottleneck)
    out = layers.Dense(input_dim)(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
def dropout_ae(input_dim):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dropout(0.3)(x)
    bottleneck = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(bottleneck)
    out = layers.Dense(input_dim)(x)
    model = models.Model(inp, out)
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
def vae(input_dim):

    latent_dim = 16

    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inputs)

    z_mean = layers.Dense(latent_dim)(x)
    z_log_var = layers.Dense(latent_dim)(x)

    def sampling(args):
        z_mean, z_log_var = args
        epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    z = layers.Lambda(sampling)([z_mean, z_log_var])

    decoder = layers.Dense(input_dim)(z)

    model = models.Model(inputs, decoder)
    model.compile(optimizer='adam', loss='mse')

    return model

In [ ]:
def improved_hybrid_rf(X_train, y_train, X_test):

    y_train_cls = (y_train >= threshold_energy).astype(int)

    root = RandomForestClassifier(
        n_estimators=1000,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )

    root.fit(X_train, y_train_cls)

    proba_test = root.predict_proba(X_test)[:,1]

    reg = RandomForestRegressor(
        n_estimators=700,
        random_state=42,
        n_jobs=-1
    )

    reg.fit(X_train, y_train)
    pred_reg = reg.predict(X_test)

    final_pred = 0.8 * pred_reg + 0.2 * proba_test * np.max(y_train)

    return final_pred

In [ ]:
def evaluate_model(name, y_test, y_pred):

    fpr, tpr, thresholds = roc_curve(y_test_cls, y_pred)
    best_thresh = thresholds[np.argmax(tpr - fpr)]

    y_pred_cls = (y_pred >= best_thresh).astype(int)

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test_cls, y_pred_cls),
        "Precision": precision_score(y_test_cls, y_pred_cls),
        "Recall": recall_score(y_test_cls, y_pred_cls),
        "F1": f1_score(y_test_cls, y_pred_cls),
        "ROC_AUC": roc_auc_score(y_test_cls, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    }

In [ ]:
results = []

ae_models = {
    "Basic": basic_ae,
    "Deep": deep_ae,
    "Denoising": denoise_ae,
    "Sparse": sparse_ae,
    "Dropout": dropout_ae,
    "VAE": vae
}

for name, builder in ae_models.items():

    model = builder(X_train.shape[1])
    X_train_enc, X_test_enc = train_autoencoder(model, name)

    # XGBoost
    xgb = XGBRegressor(n_estimators=500, learning_rate=0.05)
    xgb.fit(X_train_enc, y_train)
    pred = xgb.predict(X_test_enc)
    results.append(evaluate_model(name+" + XGB", y_test, pred))

    # Random Forest
    rf = RandomForestRegressor(n_estimators=700)
    rf.fit(X_train_enc, y_train)
    pred = rf.predict(X_test_enc)
    results.append(evaluate_model(name+" + RF", y_test, pred))

    # Hybrid
    pred = improved_hybrid_rf(X_train_enc, y_train, X_test_enc)
    results.append(evaluate_model(name+" + Hybrid", y_test, pred))

84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Basic trained.
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Deep trained.
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Denoising trained.
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Sparse trained.
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Dropout trained.
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
VAE trained.


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Accuracy", ascending=False)

print("\n========== FINAL COMPARISON ==========\n")
print(results_df)


========== FINAL COMPARISON ==========

                 Model  Accuracy  Precision    Recall        F1   ROC_AUC  \
5        Deep + Hybrid  0.909910   0.881188  0.858521  0.869707  0.949300   
1           Basic + RF  0.907658   0.893471  0.836013  0.863787  0.936363   
11     Sparse + Hybrid  0.906532   0.872549  0.858521  0.865478  0.948564   
10         Sparse + RF  0.899775   0.896429  0.807074  0.849408  0.932434   
2       Basic + Hybrid  0.899775   0.817143  0.919614  0.865356  0.954978   
17        VAE + Hybrid  0.898649   0.879725  0.823151  0.850498  0.935025   
0          Basic + XGB  0.897523   0.852564  0.855305  0.853933  0.939302   
14    Dropout + Hybrid  0.897523   0.923077  0.771704  0.840630  0.928001   
4            Deep + RF  0.896396   0.871186  0.826367  0.848185  0.930556   
6      Denoising + XGB  0.896396   0.901099  0.790997  0.842466  0.927683   
8   Denoising + Hybrid  0.895270   0.838509  0.868167  0.853081  0.943393   
13        Dropout + RF  0.893018   